<a href="https://colab.research.google.com/github/madhusudhanrao-ppm/dbdevrel/blob/main/source-codes/colab-code/similaritysearch/similaritysearch-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu "oracledb>=2.5" numpy

In [ ]:
import os
import json
from typing import List, Dict, Optional
import numpy as np
import oracledb

oracledb.defaults.program = "devrel-developerhub-similaritysearch-colab"
from sentence_transformers import SentenceTransformer
import faiss
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # not running in Colab: nothing to mount

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

def oracle_connect(use_wallet: bool = True, wallet_dir: Optional[str] = None):
    """Connect to Oracle using env vars ORACLE_USER, ORACLE_PASSWORD and ORACLE_DSN
    (an EZConnect string such as host:1521/service, or a TNS alias).
    If use_wallet=True and a wallet directory is given (wallet_dir or ORACLE_WALLET_DIR),
    the wallet is used; its password comes from ORACLE_WALLET_PASSWORD.
    Returns an oracledb connection, or None if the credentials are not set.
    """
    username = os.environ.get("ORACLE_USER")
    password = os.environ.get("ORACLE_PASSWORD")
    dsn = os.environ.get("ORACLE_DSN")

    if not (username and password and dsn):
        print("Oracle credentials (ORACLE_USER, ORACLE_PASSWORD, ORACLE_DSN) not set.")
        return None

    kwargs = {}
    wallet_dir = wallet_dir or os.environ.get("ORACLE_WALLET_DIR")
    if use_wallet and wallet_dir:
        # TNS aliases need config_dir to find tnsnames.ora
        kwargs = dict(config_dir=wallet_dir,
                      wallet_location=wallet_dir,
                      wallet_password=os.environ.get("ORACLE_WALLET_PASSWORD"))

    return oracledb.connect(user=username, password=password, dsn=dsn, **kwargs)

def fetch_from_oracle(
    table: str = "MYNOTES",
    column: str = "NOTES",
    max_rows: int = 1000,
    use_wallet: bool = True,
    wallet_dir: Optional[str] = None,
) -> List[str]:
    """Fetch text rows from Oracle. Uses env vars ORACLE_USER, ORACLE_PASSWORD, ORACLE_DSN.
    If use_wallet=True, provide wallet_dir (or ORACLE_WALLET_DIR) and ORACLE_WALLET_PASSWORD via env vars.
    Returns list of strings (may be empty).
    """
    conn = None
    try:
        conn = oracle_connect(use_wallet=use_wallet, wallet_dir=wallet_dir)
        if conn is None:
            print("Returning empty list.")
            return []

        print("✓ Successfully connected to Oracle Database")

        cur = conn.cursor()
        sql = f"SELECT {column} FROM {table} WHERE {column} IS NOT NULL AND ROWNUM <= :maxrows"
        cur.execute(sql, [max_rows])
        rows = cur.fetchall()
        texts = [r[0] for r in rows if r and r[0] is not None]
        cur.close()
        return texts
    except Exception as e:
        print(f"Error fetching from Oracle: {e}")
        return []
    finally:
        if conn:
            try:
                conn.close()
            except Exception:
                pass

def build_embeddings(texts: List[str], model_name: str = MODEL_NAME, batch_size: int = 64):
    """Return (model, numpy array of embeddings L2-normalized)."""
    model = SentenceTransformer(model_name)
    embs = model.encode(texts, batch_size=batch_size, convert_to_numpy=True, show_progress_bar=True)
    # normalize
    faiss.normalize_L2(embs)
    return model, embs

def build_faiss_index(embs: np.ndarray):
    """Create an inner-product index. Inputs should be L2-normalized vectors; inner product == cosine similarity."""
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs)
    return index

def search_index(query: str, model: SentenceTransformer, index: faiss.IndexFlatIP, texts: List[str], k: int = 5):
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k)
    # D contains inner products (cosine similarity between -1 and 1)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        results.append({"index": int(idx), "text": texts[idx], "score": float(score)})
    return results

def save_index(path_prefix: str, embs: np.ndarray, texts: List[str]):
    np.save(f"{path_prefix}_emb.npy", embs)
    with open(f"{path_prefix}_texts.json", "w", encoding="utf-8") as f:
        json.dump(texts, f, ensure_ascii=False)

def load_index(path_prefix: str):
    embs = np.load(f"{path_prefix}_emb.npy")
    with open(f"{path_prefix}_texts.json", "r", encoding="utf-8") as f:
        texts = json.load(f)
    index = build_faiss_index(embs)
    return index, embs, texts

## Create the sample MYNOTES table (optional)

`fetch_from_oracle` reads the `NOTES` column of a `MYNOTES` table, using the connection settings in the environment variables `ORACLE_USER`, `ORACLE_PASSWORD` and `ORACLE_DSN` (plus `ORACLE_WALLET_DIR` and `ORACLE_WALLET_PASSWORD` if you connect with a wallet). If you already have a `MYNOTES` table, this cell leaves it unchanged. Otherwise it creates the table with a few sample notes.

In [ ]:
# Create the sample MYNOTES table if it does not exist yet (safe to re-run)
SAMPLE_NOTES = [
    "I want to open an account.",
    "I want a credit card.",
    "I need to update my address.",
    "I want to apply for a loan.",
    "How do I check my balance?",
    "I lost my debit card.",
]

setup_conn = oracle_connect()
if setup_conn:
    with setup_conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_tables WHERE table_name = 'MYNOTES'")
        if cur.fetchone()[0] == 0:
            cur.execute("CREATE TABLE MYNOTES (NOTES VARCHAR2(1000))")
            cur.executemany("INSERT INTO MYNOTES (NOTES) VALUES (:1)", [(n,) for n in SAMPLE_NOTES])
            setup_conn.commit()
            print(f"✓ Created MYNOTES with {len(SAMPLE_NOTES)} sample notes")
        else:
            print("✓ MYNOTES table already exists - leaving it unchanged")
    setup_conn.close()

## Build the index and search

In [ ]:
# Configuration: switch to True to fetch from Oracle
use_oracle = True

texts = []
if use_oracle:
    print("Fetching from Oracle...")
    texts = fetch_from_oracle(table="MYNOTES", column="NOTES", max_rows=1000)
    print(f"Fetched {len(texts)} rows")
else:
    texts = [
        "I want to open an account.",
        "I want a credit card.",
        "I need to update my address.",
        "I want to apply for a loan.",
        "How do I check my balance?",
        "I lost my debit card."
    ]
    print(f"Using {len(texts)} sample texts")

# Build model + embeddings + index
model = None # Initialize model to None
embs = None # Initialize embs to None
index = None # Initialize index to None

if texts: # Only build embeddings and index if texts list is not empty
    model, embs = build_embeddings(texts)
    index = build_faiss_index(embs)
    print("Index ready. Use search_index(query, model, index, texts, k)")
else:
    print("No texts available to build embeddings and index.")
    print("Please check the Oracle connection details or provide sample texts.")


query = input("Enter search query: ")
if query.strip():
    results = search_index(query, model, index, texts, k=5)
    for i, r in enumerate(results, 1):
        print(f"{i}. score={r['score']:.4f}\n   {r['text'][:300]}\n")
else:
    print("No query provided.")